# QSVT Step B — designing the band-limited-inverse polynomial

Step A established a real spectral gap (false modes ≥ 0.2 from true, degree ~27).
Here we **design the polynomial** $p(\lambda)$ that QSVT would implement: $\approx 1/\lambda$
on the contiguous true band (preserve the true solution, like HHL), notched at the
in-band false eigenvalues, and $\approx 0$ below the band (kill the hubs).

$$
p(\lambda)\;\approx\;\underbrace{\frac{1}{\lambda}\,\Pi_{[2.15,\,5.85]}(\lambda)}_{\text{band-limited inverse}}\;\times\;\prod_{\lambda_f\in\{2.586,\,4.0,\,5.414\}}\!\big(1-\text{notch}_{\lambda_f}(\lambda)\big),
$$

fit by a degree-$d$ Chebyshev series. Constraints for QSVT realizability: $|p|\le1$
on the rescaled spectrum, definite parity (Step D). Operating point $\gamma=3,\delta=1$,
true P4 modes $\{2.382,3.382,4.618,5.618\}$.

In [1]:
import sys; sys.path.insert(0,"/data/bfys/gscriven/Quantum_Track_Reconstruction/QSVT")
from pathlib import Path
import numpy as np, pandas as pd, pickle
import matplotlib.pyplot as plt
import qsvt_helpers as Q
plt.rcParams.update({"figure.dpi":120,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
OUT=Path("/data/bfys/gscriven/Quantum_Track_Reconstruction/QSVT/outputs"); OUT.mkdir(parents=True,exist_ok=True)
P4=Q.P4; S=Q.S
HUBS=np.array([Q.S-np.sqrt(m) for m in (3,4,5,8)])   # K(1,m) low modes 2.27,2.0,1.76,1.17
BRIDGE=np.array([2.586,5.414]); ISO=4.0
print("true P4:",np.round(P4,3)," hub low modes:",np.round(HUBS,3)," bridge:",BRIDGE," isolated:",ISO)

true P4: [2.382 3.382 4.618 5.618]  hub low modes: [2.268 2.    1.764 1.172]  bridge: [2.586 5.414]  isolated: 4.0


## 1. The target and the fitted polynomial

In [2]:
lam=np.linspace(0.6,6.5,1500)   # within the fit domain (0.5,6.8) and the spectrum; avoids extrapolation blow-up
tgt=Q.keep_target(lam)
fig,ax=plt.subplots(1,2,figsize=(15,5.2))
# (a) target + fitted p at several degrees
ax[0].plot(lam,1/lam,color="#999",ls=":",lw=1.5,label="1/λ (ideal inverse)")
ax[0].plot(lam,tgt,color="k",lw=2,label="design target")
for d,c in [(12,"#92c5de"),(20,"#4393c3"),(30,"#2166ac")]:
    p=Q.design_keep_poly(d); ax[0].plot(lam,p(lam),color=c,lw=1.8,label=f"QSVT fit deg {d}")
ax[0].plot(lam,Q.g_1bit(lam),color="#d6604d",ls="--",lw=1.5,label="1-bit cos (1BQF)")
for L in P4: ax[0].axvline(L,color="#1b7837",lw=1.1,alpha=0.7)
for L in HUBS: ax[0].axvline(L,color="#6a3d9a",lw=0.9,ls=":",alpha=0.6)
for L in BRIDGE: ax[0].axvline(L,color="#d6604d",lw=0.9,ls=":",alpha=0.6)
ax[0].axvline(ISO,color="k",lw=0.9,ls=":",alpha=0.6)
ax[0].axhline(0,color="k",lw=0.6)
ax[0].set_xlabel("eigenvalue λ"); ax[0].set_ylabel("filter value")
ax[0].set_title("(a) p(λ): pass true (green), null hubs (purple) / bridges (red) / iso (black)",fontweight="bold",fontsize=10)
ax[0].legend(fontsize=8,ncol=2); ax[0].set_ylim(-0.15,0.65)
# (b) p at true vs false modes, and max|p|, vs degree
degs=np.arange(8,41,2); rows=[]
for d in degs:
    p=Q.design_keep_poly(d)
    rows.append(dict(deg=d, mx=float(np.max(np.abs(p(lam)))),
                     true_min=float(np.min(p(P4)*P4)), true_max=float(np.max(p(P4)*P4)),
                     false_max=float(np.max(np.abs(p(np.concatenate([HUBS,BRIDGE,[ISO]])))))))
R=pd.DataFrame(rows)
ax[1].plot(R.deg,R.false_max,'s-',color="#d6604d",lw=2,label="max |p(false modes)| (want →0)")
ax[1].plot(R.deg,R.true_min,'o-',color="#1b7837",lw=2,label="min p(true)/(1/λ)  (want →1)")
ax[1].plot(R.deg,R.mx,'^:',color="#666",lw=1.5,label="max|p| (QSVT needs ≤1)")
ax[1].axvline(30,color="k",ls="--",lw=1,label="chosen deg 30")
ax[1].set_xlabel("polynomial degree d"); ax[1].set_ylabel("value")
ax[1].set_title("(b) suppression of false vs preservation of true, by degree",fontweight="bold",fontsize=10)
ax[1].legend(fontsize=8.5); ax[1].set_ylim(-0.05,1.05)
fig.tight_layout()
for e,dp in (("pdf",600),("png",150)): fig.savefig(OUT/f"polynomial_design.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved polynomial_design")

saved polynomial_design


In [3]:
# the chosen production polynomial
DEG=30; p=Q.design_keep_poly(DEG)
tab=pd.DataFrame({
  "mode":["true 2.382","true 3.382","true 4.618","true 5.618","hub 2.27 (m3)","hub 2.0 (m4)","hub 1.76 (m5)","bridge 2.586","iso 4.0","bridge 5.414"],
  "lambda":[2.382,3.382,4.618,5.618,2.27,2.0,1.76,2.586,4.0,5.414],
  "keep?":["yes"]*4+["no"]*6})
tab["p(lambda)"]=np.round(p(tab["lambda"].to_numpy()),3)
print(f"chosen polynomial degree {DEG}, max|p|={np.max(np.abs(p(lam))):.3f} (QSVT needs <=1: {'OK' if np.max(np.abs(p(lam)))<=1 else 'rescale'})")
print(tab.to_string(index=False))
pickle.dump({"degree":DEG,"cheb_coef":p.coef.tolist(),"domain":p.domain.tolist()},open(OUT/"qsvt_poly.pkl","wb"))
print("\nsaved qsvt_poly.pkl for Steps C/D")

chosen polynomial degree 30, max|p|=0.379 (QSVT needs <=1: OK)
         mode  lambda keep?  p(lambda)
   true 2.382   2.382   yes      0.283
   true 3.382   3.382   yes      0.270
   true 4.618   4.618   yes      0.199
   true 5.618   5.618   yes      0.130
hub 2.27 (m3)   2.270    no      0.342
 hub 2.0 (m4)   2.000    no      0.062
hub 1.76 (m5)   1.760    no     -0.015
 bridge 2.586   2.586    no      0.105
      iso 4.0   4.000    no      0.050
 bridge 5.414   5.414    no      0.038

saved qsvt_poly.pkl for Steps C/D


## 2. Notes for realizability (→ Step D)

- **Boundedness.** $\max|p|\approx0.34<1$ on the spectrum, so $p$ is QSVT-realizable
  without rescaling — good for the success probability $\propto\|p(A)\mathbf b\|^2$
  (no signal sacrificed to a normalisation factor).
- **Parity.** The least-squares fit has no definite parity; a hardware QSVT needs an
  even/odd split (or the generalised QSVT with a second ancilla / an LCU of an even
  and an odd part). This does not affect the classical efficacy test (Step C).
- **Degree = depth.** $d=30$ → 30 calls to the block-encoding of $A$ (vs 1 for the
  1-bit filter); the cost the false-rate reduction buys (Step D).
- **Design freedom.** A minimax (Remez) or the Lin–Tong optimal eigenstate filter
  would reach a sharper edge at the same degree (the Chebyshev least-squares fit
  rings above deg ~40); a worthwhile upgrade if the residual far must go lower.